# Step 3 - BEV 투영 및 음영(A_shadow) 정량 계측 + 탑뷰 시각화

**목표 (이번 PoC의 핵심 산출물)**: Step 1(구조물 마스크 + 지면 마스크) + Step 2(깊이) + Step 0(카메라 K)를 결합해
1. Front view를 **탑뷰(BEV) 점유 격자**로 투영 (지면/하늘 픽셀은 제외 = 옵션 A)
2. 차량(원점)에서 전방 레이캐스팅으로 **Obstacle 뒤 Shadow (blind zone) 다각형** 추출
3. A_shadow(m^2), L_vis(m) 계산 및 **탑뷰 음영 시각화 이미지** 저장

**L_vis 정의**: 정면 ±10° 레이가 처음 구조물에 닿는 거리의 중앙값(BEV 레이캐스팅 기반).
02의 이미지 ROI 방식은 도로 바닥을 잡아 부정확해 제거했고, 본 단계 한 곳에서만 산출한다.

**입력**: `output/00_front/*_cam.json`, `output/01_seg/` (`masks`, `ground_mask`), `output/02_depth/`

**출력**: `output/03_bev/{stem}_bev.jpg`, `{stem}_dsi.json`

> 제안서 4~5단계(D_stopping/V_heavy/실시간 API)는 이번 범위 밖. 합격 기준은 **음영 시각화**이며
> A_shadow/L_vis 절대값은 단안 깊이 근사라 상대 지표로만 취급한다.

## 0. 패키지 설치

In [1]:
# !pip install opencv-python-headless numpy matplotlib

## 1. 라이브러리 및 BEV 파라미터

In [ ]:
import json
import math
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

FRONT_DIR = Path("test_output/00_front")
SEG_DIR   = Path("test_output/01_seg")
DEPTH_DIR = Path("test_output/02_depth")
OUT_DIR   = Path("test_output/03_bev")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_PATHS = sorted(FRONT_DIR.glob("*_front.jpg"))
print(f"Front view {len(IMG_PATHS)}장")

TEST_MODE = False   # True: 무작위 10장만 처리(테스트) / False: 전체 처리
TEST_N = 10
TEST_SEED = 42
if TEST_MODE:
    IMG_PATHS = sorted(random.Random(TEST_SEED).sample(IMG_PATHS, min(TEST_N, len(IMG_PATHS))))
    print(f"[TEST_MODE] 무작위 {len(IMG_PATHS)}장만 처리")

RESUME = False and not TEST_MODE   # True: 이미 처리된 파일 건너뜀 / False: 전체 재처리 (테스트 모드는 항상 재처리)

# BEV 격자 파라미터
GRID_RES   = 0.5     # m / pixel
MAX_RANGE  = 60.0    # 전방 최대 거리 (m)
HALF_WIDTH = 30.0    # 좌우 폭 (m)
GRID_H = int(MAX_RANGE / GRID_RES)
GRID_W = int(HALF_WIDTH * 2 / GRID_RES)
ORIGIN = (GRID_W // 2, GRID_H - 1)   # (col,row): 하단 중앙 = 차량

CAM_HEIGHT = 2.5    # 스트리트뷰 카메라 높이(m) 가정 (IPM 지면접점 투영용)

# 촬영 차량 본네트(ego-vehicle) 제외: 프론트 뷰 하단은 항상 촬영 차량 본네트라 car로 분류돼
# vehicle_mask에 들어감. 이 비율만큼의 하단 영역을 BEV 투영에서 제외해 원점 앞 본네트가
# occluder로 찍히는 것을 막는다. 고정 리그라 본네트 위치가 일정해 고정 비율로 처리. 0이면 비활성.
HOOD_MASK_FRAC = 0.25  # 하단 25%

# 차량 분류: 전방 중앙(±CENTER_HALF_ANGLE_DEG) 안의 차량은 촬영차와 같이 움직이는 co-moving으로 보고
# occluder에서 제외(표시만). 그 밖(측면)에 있는 차량만 시야 차단 occluder로 취급한다.
# (보행자 갑툭튀 등 이동 위험은 측면에서 발생 / 정면 선행차 overcap 방지)
CENTER_HALF_ANGLE_DEG = 15.0

# INCLUDE_VEHICLES: True면 '건물만' vs '건물+측면차량' 비교 패널을 함께 출력.
# (중앙 co-moving 차량은 양쪽 모두에서 occluder 아님)
INCLUDE_VEHICLES = True

# --- Method A 검증 레이어 (leakage 경계 = emergence 위험) ---
# "가려진 면적"이 아니라 "시선이 occluder 가장자리를 스쳐 지나가는 곳"(골목 입구·차 사이 틈 등)을
# 위험으로 본다. 인접 레이 hit 거리의 불연속으로 가장자리를 검출 → 연속 벽면은 매끄러워 안 잡힘.
# DSI에는 아직 반영하지 않고, 시각화 패널 + H_leak 지표만 추가(검증용).
SHOW_HAZARD    = True   # True: hazard(emergence) 패널 추가 / False: 기존 레이아웃
HAZ_GAP_THRESH = 3.0    # 인접 레이 hit 거리차가 이 값(m) 이상이면 가장자리(emergence)로 간주
HAZ_R          = 40.0   # 위험 가중 기준거리(m). 이 안쪽일수록 위험↑(반응시간 짧음)

print(f"BEV 격자: {GRID_W} x {GRID_H} px ({GRID_RES} m/px), 전방 {MAX_RANGE}m, 좌우 +-{HALF_WIDTH}m")
print(f"중앙 cone +-{CENTER_HALF_ANGLE_DEG}° 차량=co-moving | 비교(INCLUDE_VEHICLES): {INCLUDE_VEHICLES} | Hazard: {SHOW_HAZARD}")


## 2. 마스크 -> BEV 점유 격자 (평지 IPM, 지면접점)

**방식 변경 (단안 깊이 → IPM)**: 기존엔 각 픽셀을 `depth_norm*SCALE` 광선거리로 3D에 찍어 BEV에 놓았는데,
단안 깊이가 절대 스케일이 없고 수직 벽의 중간 높이 픽셀이 멀리 찍혀 **가까운 벽이 바깥·앞으로 퍼지는**
문제가 있었다(우측 과도 낙관). 이를 **IPM(Inverse Perspective Mapping)** 으로 교체한다.

- 평지·`pitch≈0`·카메라 높이 `CAM_HEIGHT` 가정.
- 각 **열(column)** 에서 장애물의 **지면 접점 = 최하단 장애물 픽셀** `(u, v_base)` 하나만 사용.
- 광선이 지면(`Y=CAM_HEIGHT`)에 닿는 거리로 역투영: `Z = CAM_HEIGHT * fy / (v_base - cy)`, `X = (u-cx)/fx * Z`.
- 단안 깊이를 거리로 쓰지 않으므로 벽 발자취가 실제 지면 위치에 정확히 찍힌다.

지면/하늘(`ground_mask`)·본네트 영역은 접점 탐색에서 제외한다. 열 간 간격으로 레이가 새지 않도록
발자취를 3x3로 살짝 팽창한다. (깊이는 시각화 패널 용도로만 유지)

In [ ]:
def _column_base_z(col_mask, cy, fy, cam_height):
    """열 마스크의 최하단 픽셀(지면 접점)을 IPM 거리(m)로. 없거나 범위 밖이면 nan."""
    rows = np.where(col_mask)[0]
    if not len(rows):
        return np.nan
    yn = (int(rows.max()) - cy) / fy
    if yn <= 1e-3:                      # 지평선 위 → 지면과 교차 안 함
        return np.nan
    Z = cam_height / yn
    return Z if 0 < Z <= MAX_RANGE else np.nan


def split_vehicles_by_angle(veh_mask, K, center_half_deg, min_area=50):
    """차량 마스크를 개체(connected component) 단위로 나눠 각 차량을 전방 각도로 분류.
    centroid x의 각도 |atan((x-cx)/fx)| < center_half_deg → 중앙(co-moving, 비occluder),
    그 외 → 측면(occluder). min_area 미만 컴포넌트는 노이즈로 무시.
    반환: (side_mask, center_mask) — 둘 다 bool, 입력과 동일 크기."""
    fx, cx = K[0, 0], K[0, 2]
    side   = np.zeros_like(veh_mask, dtype=bool)
    center = np.zeros_like(veh_mask, dtype=bool)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(veh_mask.astype(np.uint8))
    for lab in range(1, n):                       # 0 = 배경
        if stats[lab, cv2.CC_STAT_AREA] < min_area:
            continue
        ang = math.degrees(math.atan((centroids[lab][0] - cx) / fx))
        if abs(ang) < center_half_deg:
            center |= (labels == lab)
        else:
            side |= (labels == lab)
    return side, center


def mask_to_footprint(mask, K, cam_height, subsample=2, dilate=True,
                      connect=True, smooth_win=5, connect_thresh=4.0, cap_mask=None):
    """평지 IPM 지면접점 발자취. 각 열의 최하단 픽셀을 역투영하되, 끊긴 점이 아니라
    연속 발자취를 만든다:
      (1) 열 방향 Z 프로파일을 median 평활 → 차/라벨 노이즈로 v_base가 튀어 생기는 거리 스파이크 제거.
      (2) 인접 열의 투영점을 |ΔZ| < connect_thresh 일 때 선분으로 연결 → 연속 벽은 메우고,
          큰 점프(골목 입구·차 사이 틈)는 잇지 않아 '진짜 가장자리'로 남긴다.
    cap_mask(전경, 예: 측면 차량)가 주어지면, 그 열의 전경이 mask보다 가까울 때 거리를 전경 거리로 캡한다.
    → 주차 차량이 건물 밑동을 가려 건물이 '차 뒤 먼 벽'으로 지어지는(false alley·leak) 문제 방지.
    (pitch≈0, 평지, cam_height 가정)"""
    h, w = mask.shape
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    us = list(range(0, w, subsample))

    # 1) 열별 지면접점 거리 Z (없거나 범위 밖이면 nan). 전경이 더 가까우면 그 거리로 캡.
    z = np.full(len(us), np.nan, dtype=np.float32)
    for i, u in enumerate(us):
        Zi = _column_base_z(mask[:, u], cy, fy, cam_height)
        if not np.isfinite(Zi):
            continue
        if cap_mask is not None:
            Zc = _column_base_z(cap_mask[:, u], cy, fy, cam_height)
            if np.isfinite(Zc) and Zc < Zi:   # 전경이 더 가까움 → 그 거리에서 시야 차단
                Zi = Zc
        z[i] = Zi

    # 2) median 평활 (유한값만, 중심이 유효한 열에 한해)
    if smooth_win > 1:
        k = smooth_win // 2
        zs = z.copy()
        for i in range(len(z)):
            if not np.isfinite(z[i]):
                continue
            seg = z[max(0, i - k): i + k + 1]
            seg = seg[np.isfinite(seg)]
            if len(seg):
                zs[i] = np.median(seg)
        z = zs

    # 3) 투영 + 인접 열 연결
    canvas = np.zeros((GRID_H, GRID_W), dtype=np.uint8)
    prev = None   # (i, col, row)
    for i, u in enumerate(us):
        Zi = z[i]
        if not np.isfinite(Zi):
            prev = None
            continue
        X = (u - cx) / fx * Zi
        col = int(round(ORIGIN[0] + X / GRID_RES))
        row = int(round(ORIGIN[1] - Zi / GRID_RES))
        if not (0 <= row < GRID_H and 0 <= col < GRID_W):
            prev = None
            continue
        canvas[row, col] = 1
        if connect and prev is not None and (i - prev[0]) == 1 \
                and abs(Zi - z[prev[0]]) < connect_thresh:
            cv2.line(canvas, (prev[1], prev[2]), (col, row), 1, 1)
        prev = (i, col, row)

    if dilate:   # 레이가 새지 않도록 살짝 팽창
        canvas = cv2.dilate(canvas, np.ones((3, 3), np.uint8))
    return canvas.astype(bool)


print("BEV 투영(IPM) 함수 정의 완료")

## 3. 레이캐스팅 -> 음영 다각형

원점(차량)에서 전방 섹터로 레이를 쏘아 Obstacle에 처음 닿은 뒤쪽을 음영으로 표시.
레이 각도 범위 = 정면 핀홀 HFOV 와 일치(빈 격자만 지나 A_shadow=0 되는 문제 방지).

In [4]:
def raycast_shadow(grid_occ, hfov_deg):
    """원점에서 전방 섹터로 레이를 쏘아 첫 occluder 뒤를 음영으로 표시."""
    half = hfov_deg / 2.0
    ray_angles = np.linspace(-half, half, int(hfov_deg * 2) + 1)
    shadow_grid = np.zeros((GRID_H, GRID_W), dtype=bool)
    ray_hits = []
    ox, oy = ORIGIN
    step = GRID_RES * 0.7
    max_steps = int(MAX_RANGE / step)

    for ang in ray_angles:
        a = math.radians(ang)
        dx = math.sin(a)     # col 방향
        dy = -math.cos(a)    # row 방향(전방 = row 감소)
        hit = MAX_RANGE
        behind = False
        for s in range(max_steps):
            dist = s * step
            col = int(round(ox + dx * dist / GRID_RES))
            row = int(round(oy + dy * dist / GRID_RES))
            if col < 0 or col >= GRID_W or row < 0 or row >= GRID_H:
                break
            if not behind and grid_occ[row, col]:
                hit = dist
                behind = True
            if behind:
                shadow_grid[row, col] = True
        ray_hits.append((ang, hit))
    return shadow_grid, ray_hits


def l_vis_from_rays(ray_hits):
    front = [d for (a, d) in ray_hits if -10 <= a <= 10]
    return float(np.median(front)) if front else MAX_RANGE


def detect_emergence(ray_hits, r_haz, gap_thresh):
    """[Method A] 인접 레이 hit 거리의 불연속(occluder 가장자리)을 찾아 emergence(유입) 지점 추출.
    연속 벽면은 거리가 매끄러워 안 잡히고, 골목 입구/차 사이 틈처럼 시선이 가장자리를 스쳐
    지나가는 곳만 잡힌다. 가까운 쪽 가장자리를 유입점으로 보고 근접 가중치를 부여한다.
    반환: edges=[(angle_mid_deg, dist_m, weight)], h_leak(Σweight), n_edges."""
    edges = []
    h_leak = 0.0
    for (a0, d0), (a1, d1) in zip(ray_hits[:-1], ray_hits[1:]):
        if abs(d1 - d0) < gap_thresh:        # 거리 변화 작음 = 연속면 → 무시
            continue
        d_edge = min(d0, d1)                  # 가까운 쪽 가장자리 = 유입점
        if d_edge >= MAX_RANGE:               # 둘 다 비었으면(가림 없음) 무시
            continue
        w = max(0.0, 1.0 - d_edge / r_haz)    # 가까울수록 위험↑
        if w <= 0:
            continue
        edges.append(((a0 + a1) / 2.0, d_edge, w))
        h_leak += w
    return edges, h_leak, len(edges)


print("레이캐스팅 / emergence 함수 정의 완료")

레이캐스팅 / emergence 함수 정의 완료


## 4. A_shadow / DSI_refined 기초값

In [5]:
def compute_dsi_refined(l_vis, a_shadow, a_total, v_heavy=0.0, speed_kmh=50.0):
    """DSI_refined = (D_stopping/L_vis) * (1 + A_shadow/A_total) * (1 + V_heavy).
    (제안서 4단계 변수는 PoC 기본값. 절대값보다 음영 시각화가 본 검증 목표)"""
    v = speed_kmh / 3.6
    mu, g = 0.7, 9.8
    d_stop = v * 1.0 + v * v / (2 * mu * g)   # 반응 1s + 제동
    l_vis = max(l_vis, 0.1)
    dsi = (d_stop / l_vis) * (1 + a_shadow / max(a_total, 1)) * (1 + v_heavy)
    return dsi, d_stop


def grade_of(dsi):
    return "Safe" if dsi < 1.0 else ("Caution" if dsi < 1.8 else "High-risk")


A_TOTAL = MAX_RANGE * HALF_WIDTH * 2
print(f"A_total(이론): {A_TOTAL:.0f} m^2")

A_total(이론): 3600 m^2


## 5. 전체 파이프라인 실행 + 탑뷰 시각화

In [ ]:
from tqdm.auto import tqdm

SPEED_LIMIT_KMH = 50.0
V_HEAVY = 0.15   # PoC 기본값(중차량 비율 4단계 미정)

# 공통 legend 핸들 (루프 밖 1회 정의)
leg_bg      = mpatches.Patch(color=[200/255,200/255,200/255], label="No data")
leg_other   = mpatches.Patch(color=[100/255,200/255,100/255], label="Other (non-blocker)")
leg_occ     = mpatches.Patch(color=[220/255,80/255,80/255],   label="Obstacle (blocker)")
leg_veh     = mpatches.Patch(color=[1, 140/255, 0],           label="Vehicle (side, blocker)")
leg_veh_co  = mpatches.Patch(color=[120/255, 170/255, 1],     label="Vehicle (co-moving)")
leg_shadow  = mpatches.Patch(color=[1, 0.78, 0],              label="Shadow (blind zone)")
leg_em      = plt.Line2D([0], [0], marker="o", color="magenta", linestyle="None",
                         markersize=8, label="Emergence edge (risk)")
leg_vehicle = plt.Line2D([0], [0], marker="^", color="blue", markersize=8,
                         linestyle="None", label="Vehicle (origin)")


def render_shadow(ax, bev, shadow_grid, ray_hits, title, legend_handles):
    srgb = bev.copy(); srgb[shadow_grid] = [255, 200, 0]
    ax.imshow(srgb, origin="upper")
    stepn = max(1, len(ray_hits) // 12)
    for ang, hd in ray_hits[::stepn]:
        a = math.radians(ang)
        ax.annotate("", xy=(ORIGIN[0] + math.sin(a)*hd/GRID_RES,
                            ORIGIN[1] - math.cos(a)*hd/GRID_RES),
                    xytext=ORIGIN, arrowprops=dict(arrowstyle="->", color="blue", lw=0.7))
    ax.plot(ORIGIN[0], ORIGIN[1], "b^", markersize=10)
    ax.set_title(title, fontsize=9); ax.axis("off")
    ax.legend(handles=legend_handles, loc="upper right", fontsize=8)


def render_hazard(ax, bev, shadow_grid, edges, title):
    hbg = bev.copy(); hbg[shadow_grid] = [255, 235, 150]   # 전체 shadow는 옅은 노랑(맥락용)
    ax.imshow(hbg, origin="upper")
    for ang, d, wgt in edges:                              # emergence 가장자리 = 마젠타 점
        a = math.radians(ang)
        ex = ORIGIN[0] + math.sin(a) * d / GRID_RES
        ey = ORIGIN[1] - math.cos(a) * d / GRID_RES
        ax.scatter([ex], [ey], s=15 + 90*wgt, c="magenta", edgecolors="k",
                   linewidths=0.3, zorder=5)
    ax.plot(ORIGIN[0], ORIGIN[1], "b^", markersize=10)
    ax.set_title(title, fontsize=9); ax.axis("off")
    ax.legend(handles=[leg_em, leg_occ, leg_veh, leg_veh_co, leg_other, leg_bg, leg_vehicle],
              loc="upper right", fontsize=8)


dsi_summary = []

# 이미 처리된 파일 로드 (RESUME=True일 때만)
done = set()
if RESUME:
    for p in OUT_DIR.glob("*_dsi.json"):
        stem = p.stem.replace("_dsi", "")
        dsi_summary.append(json.load(open(p, encoding="utf-8")))
        done.add(stem)

remaining = [p for p in IMG_PATHS if p.stem not in done]
print(f"이미 완료: {len(done)}장 / 전체: {len(IMG_PATHS)}장 → 남은 작업: {len(remaining)}장")

for img_path in tqdm(remaining, desc="BEV 처리"):
    stem = img_path.stem

    cam_path  = FRONT_DIR / (stem.replace("_front", "") + "_cam.json")
    depth_npz = DEPTH_DIR / (stem + "_depth.npz")
    seg_npz   = SEG_DIR   / (stem + "_masks.npz")
    img_bgr = cv2.imread(str(img_path)); h, w = img_bgr.shape[:2]

    if not cam_path.exists() or not depth_npz.exists():
        print(f"  [SKIP] {img_path.name}: cam.json 또는 depth.npz 없음 (00/02 먼저 실행)"); continue
    cam = json.load(open(cam_path, encoding="utf-8"))
    K = np.array(cam["K"]); hfov = cam["hfov_deg"]
    with np.load(depth_npz) as dz:
        depth_norm = dz["depth_norm"].copy()

    if seg_npz.exists():
        with np.load(seg_npz) as seg_data:
            seg_masks = seg_data["masks"].copy()
            ground_mask = seg_data["ground_mask"].copy() if "ground_mask" in seg_data.files \
                else np.zeros((h, w), bool)
            vehicle_mask = seg_data["vehicle_mask"].copy() if "vehicle_mask" in seg_data.files \
                else np.zeros((h, w), bool)
    else:
        seg_masks = np.zeros((0, h, w), dtype=bool)
        ground_mask = np.zeros((h, w), bool)
        vehicle_mask = np.zeros((h, w), bool)
    obstacle_mask = seg_masks.any(axis=0) if len(seg_masks) else np.zeros((h, w), bool)

    # 촬영 차량 본네트(ego-vehicle) 제외: 하단 HOOD_MASK_FRAC 영역을 지면 마스크에 편입해
    # BEV 투영에서 빼버린다 → 원점 앞 본네트가 occluder로 찍혀 생기는 가짜 점유/음영 방지.
    if HOOD_MASK_FRAC > 0:
        ground_mask[int(h * (1 - HOOD_MASK_FRAC)):, :] = True

    # 차량을 측면(occluder) / 전방중앙(co-moving, 비occluder)으로 개체 단위 분류
    valid = ~ground_mask
    veh_valid = vehicle_mask & valid
    veh_side, veh_center = split_vehicles_by_angle(veh_valid, K, CENTER_HALF_ANGLE_DEG)

    # 지면접점 IPM 발자취. 구조물(occ)은 '측면 차량'으로만 거리 캡(중앙 선행차 overcap 방지).
    # occ/측면차/중앙차는 평활+연결로 연속화, 기타(road)는 점만(시각화용).
    other_mask = valid & (~obstacle_mask) & (~vehicle_mask)
    grid_occ        = mask_to_footprint(obstacle_mask & valid, K, CAM_HEIGHT, subsample=2, cap_mask=veh_side)
    grid_veh_side   = mask_to_footprint(veh_side,   K, CAM_HEIGHT, subsample=2)
    grid_veh_center = mask_to_footprint(veh_center, K, CAM_HEIGHT, subsample=2)
    grid_road       = mask_to_footprint(other_mask, K, CAM_HEIGHT, subsample=2, dilate=False, connect=False)

    # 건물만 occluder (항상 계산 = baseline)
    shadow_occ, ray_hits_occ = raycast_shadow(grid_occ, hfov)
    l_vis_s = l_vis_from_rays(ray_hits_occ)
    a_shadow_s = shadow_occ.sum() * (GRID_RES ** 2)
    dsi_s, d_stop = compute_dsi_refined(l_vis_s, a_shadow_s, A_TOTAL, V_HEAVY, SPEED_LIMIT_KMH)
    grade_s = grade_of(dsi_s)

    rec = {"image": img_path.name, "l_vis_m": round(l_vis_s, 2),
        "d_stopping_m": round(d_stop, 2), "a_shadow_m2": round(a_shadow_s, 2),
        "a_total_m2": A_TOTAL, "v_heavy": V_HEAVY, "dsi_refined": round(dsi_s, 4), "grade": grade_s,
        "include_vehicles": INCLUDE_VEHICLES}

    # 측면 차량 포함 시: 건물+측면차를 occluder로 한 음영을 추가 계산(비교용)
    if INCLUDE_VEHICLES:
        shadow_veh, ray_hits_veh = raycast_shadow(grid_occ | grid_veh_side, hfov)
        l_vis_v = l_vis_from_rays(ray_hits_veh)
        a_shadow_v = shadow_veh.sum() * (GRID_RES ** 2)
        dsi_v, _ = compute_dsi_refined(l_vis_v, a_shadow_v, A_TOTAL, V_HEAVY, SPEED_LIMIT_KMH)
        grade_v = grade_of(dsi_v)
        rec.update({"l_vis_veh_m": round(l_vis_v, 2), "a_shadow_veh_m2": round(a_shadow_v, 2),
                    "dsi_veh": round(dsi_v, 4), "grade_veh": grade_v})

    # [Method A] emergence(leakage) 위험: 현재 활성 occluder(측면차 포함이면 +side)의 레이로 검출
    ray_hits_haz = ray_hits_veh if INCLUDE_VEHICLES else ray_hits_occ
    shadow_haz   = shadow_veh   if INCLUDE_VEHICLES else shadow_occ
    edges, h_leak, n_edges = detect_emergence(ray_hits_haz, HAZ_R, HAZ_GAP_THRESH)
    rec.update({"h_leak": round(h_leak, 3), "hazard_edges": n_edges, "haz_r": HAZ_R})
    dsi_summary.append(rec)

    # 패널 0: 세그멘테이션 결과 (없으면 front view 폴백)
    seg_vis_path = SEG_DIR / f"{stem}_seg_vis.jpg"
    panel0 = cv2.resize(cv2.imread(str(seg_vis_path if seg_vis_path.exists() else img_path)),
                        (w, h))
    panel0_title = "Segmentation (blockers)" if seg_vis_path.exists() else "Front view"

    # 패널 1: depth 컬러맵 단독 이미지
    depth_vis_path = DEPTH_DIR / f"{stem}_depth_vis.jpg"
    if depth_vis_path.exists():
        panel1 = cv2.resize(cv2.imread(str(depth_vis_path)), (w, h))
    else:
        panel1 = (cm.plasma(depth_norm)[:, :, :3] * 255).astype(np.uint8)
        panel1 = cv2.cvtColor(panel1, cv2.COLOR_RGB2BGR)

    # 패널 2: BEV occupancy
    bev = np.ones((GRID_H, GRID_W, 3), np.uint8) * 200
    bev[grid_road]        = [100, 200, 100]
    bev[grid_occ]         = [220, 80, 80]
    bev[grid_veh_center]  = [120, 170, 255]   # 중앙 차량 (co-moving, 옅은 파랑)
    bev[grid_veh_side]    = [255, 140, 0]     # 측면 차량 (occluder, 주황)

    # 패널 수: base 3 + 음영(1 or 2) + hazard(0 or 1)
    n_shadow = 2 if INCLUDE_VEHICLES else 1
    n_panels = 3 + n_shadow + (1 if SHOW_HAZARD else 0)
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    axes[0].imshow(cv2.cvtColor(panel0, cv2.COLOR_BGR2RGB))
    axes[0].set_title(panel0_title, fontsize=9); axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(panel1, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Depth map (brighter = farther)", fontsize=9); axes[1].axis("off")

    axes[2].imshow(bev, origin="upper")
    axes[2].plot(ORIGIN[0], ORIGIN[1], "b^", markersize=10)
    axes[2].set_title(f"BEV occupancy grid ({GRID_RES}m/px)", fontsize=9); axes[2].axis("off")
    axes[2].legend(handles=[leg_occ, leg_veh, leg_veh_co, leg_other, leg_bg, leg_vehicle],
                   loc="upper right", fontsize=8)

    ax_i = 3
    if INCLUDE_VEHICLES:
        render_shadow(axes[ax_i], bev, shadow_occ, ray_hits_occ,
                      f"Shadow (buildings only)\nL_vis={l_vis_s:.1f}m  A_shadow={a_shadow_s:.0f}m^2  DSI={dsi_s:.2f} [{grade_s}]",
                      [leg_shadow, leg_occ, leg_veh_co, leg_other, leg_bg, leg_vehicle]); ax_i += 1
        render_shadow(axes[ax_i], bev, shadow_veh, ray_hits_veh,
                      f"Shadow (+side vehicles)\nL_vis={l_vis_v:.1f}m  A_shadow={a_shadow_v:.0f}m^2  DSI={dsi_v:.2f} [{grade_v}]",
                      [leg_shadow, leg_occ, leg_veh, leg_veh_co, leg_other, leg_bg, leg_vehicle]); ax_i += 1
        plt.suptitle(f"{stem[:40]}  |  buildings DSI={dsi_s:.3f} [{grade_s}]   vs   +side-veh DSI={dsi_v:.3f} [{grade_v}]",
                     fontsize=11, y=1.01)
    else:
        render_shadow(axes[ax_i], bev, shadow_occ, ray_hits_occ,
                      f"Shadow (yellow)  L_vis={l_vis_s:.1f}m\nA_shadow={a_shadow_s:.0f}m^2  DSI={dsi_s:.2f} [{grade_s}]",
                      [leg_shadow, leg_occ, leg_veh, leg_veh_co, leg_other, leg_bg, leg_vehicle]); ax_i += 1
        plt.suptitle(f"{stem[:40]}  |  DSI={dsi_s:.3f} [{grade_s}]", fontsize=11, y=1.01)

    if SHOW_HAZARD:
        haz_tag = "+side veh" if INCLUDE_VEHICLES else "buildings"
        render_hazard(axes[ax_i], bev, shadow_haz, edges,
                      f"Hazard: emergence edges ({haz_tag})\nH_leak={h_leak:.2f}  n={n_edges}  (R_HAZ={HAZ_R:.0f}m)")
        ax_i += 1

    plt.tight_layout()
    fig.savefig(str(OUT_DIR / (stem + "_bev.jpg")), dpi=120, bbox_inches="tight")
    plt.close(fig)

    json.dump(rec, open(OUT_DIR / (stem + "_dsi.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)

    # 루프 내 큰 배열 명시 해제
    del depth_norm, seg_masks, ground_mask, obstacle_mask, vehicle_mask, other_mask
    del valid, veh_valid, veh_side, veh_center
    del grid_occ, grid_road, grid_veh_side, grid_veh_center, shadow_occ, bev, panel0, panel1, img_bgr
    if INCLUDE_VEHICLES:
        del shadow_veh

print("\n=== 전체 완료 ===")

## 6. 요약

In [ ]:
json.dump(dsi_summary, open(OUT_DIR / "dsi_summary.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

header = f"{"이미지":<32}{"L_vis":>7}{"A_shadow":>10}{"DSI":>7}  등급"
print(f"=== BEV/DSI 요약 ({len(dsi_summary)}장) ===")
print(header)
print("-" * 62)
for r in dsi_summary:
    print(f"{r['image'][:31]:<32}{r['l_vis_m']:>6.1f}m{r['a_shadow_m2']:>8.0f}m^2{r['dsi_refined']:>7.2f}  [{r['grade']}]")
print(f"\n저장 위치: {OUT_DIR}")

이미지                               L_vis  A_shadow    DSI  등급
--------------------------------------------------------------
cut_point_1000_pano_4-zgBUPlEx-   46.9m    1500m^2   0.97  [Safe]
cut_point_1001_pano_IQ4H3AfYxt5   43.4m    1346m^2   1.02  [Caution]
cut_point_1005_pano_sSNXY4Ndp-C   45.5m    1410m^2   0.98  [Safe]
cut_point_1006_pano_Rr8L4nop0nW   44.1m    1687m^2   1.07  [Caution]
cut_point_1007_pano_JCJoTj97-ps   44.5m    1555m^2   1.04  [Caution]
cut_point_1008_pano_aHMpHsJEwWH   42.4m    1567m^2   1.09  [Caution]
cut_point_1009_pano_XeD5q-0U1WD   46.5m    1303m^2   0.94  [Safe]
cut_point_100_pano_3h8UuIgEVhDp   46.9m    1515m^2   0.97  [Safe]
cut_point_1010_pano_oaDIYeTodpy   46.9m    1216m^2   0.92  [Safe]
cut_point_1014_pano_4IdLIexntFa   43.4m    1646m^2   1.08  [Caution]
cut_point_1016_pano_KD_eTNUuea9   42.7m    1600m^2   1.09  [Caution]
cut_point_101_pano_-leTuFvUyJTv   46.9m    1025m^2   0.88  [Safe]
cut_point_1026_pano_tnNrfDc7Hb_   38.5m    1530m^2   1.19  [Cautio

In [ ]:
sorted_dsi = sorted(dsi_summary, key=lambda r: r["dsi_refined"], reverse=True)

print("=== DSI 상위 5 (위험도 높음) ===")
print(f"{'순위':<4} {'이미지':<35} {'DSI':>6}  등급")
print("-" * 55)
for i, r in enumerate(sorted_dsi[:5], 1):
    print(f"{i:<4} {r['image'][:34]:<35} {r['dsi_refined']:>6.3f}  [{r['grade']}]")

print()
print("=== DSI 하위 5 (위험도 낮음) ===")
print(f"{'순위':<4} {'이미지':<35} {'DSI':>6}  등급")
print("-" * 55)
for i, r in enumerate(reversed(sorted_dsi[-5:]), 1):
    print(f"{i:<4} {r['image'][:34]:<35} {r['dsi_refined']:>6.3f}  [{r['grade']}]")

=== DSI 상위 5 (위험도 높음) ===
순위   이미지                                    DSI  등급
-------------------------------------------------------
1    cut_point_201_pano_mYY_be6KWITY4dQ  156.471  [High-risk]
2    cut_point_751_pano_ufnuWtMTY_AZa4Y  156.471  [High-risk]
3    cut_point_848_pano_Eqv-txMWHmFnTtk  156.471  [High-risk]
4    cut_point_191_pano__pX3ZZRJuKvc5eN  156.458  [High-risk]
5    cut_point_467_pano_yNJYeSnaVKvaDox  156.445  [High-risk]

=== DSI 하위 5 (위험도 낮음) ===
순위   이미지                                    DSI  등급
-------------------------------------------------------
1    cut_point_1551_pano_ZRc0qYAVaKel2L   0.546  [Safe]
2    cut_point_893_pano_f8dl4oSXhgXQhmd   0.573  [Safe]
3    cut_point_1135_pano_oIUJLA2bpJSyg6   0.614  [Safe]
4    cut_point_1621_pano_MJcZwGJq6k6Wyh   0.620  [Safe]
5    cut_point_19_pano_ld__-DjkYyIIsD3w   0.622  [Safe]
